# Notebook Orquestador del Pipeline de Credit Risk
Este notebook permite ejecutar **todo el flujo MLOps** del proyecto:
- Carga de configuración
- Lectura de datos
- Exploración inicial
- Entrenamiento de modelos (XGBoost, Regresión Logística, MLP)
- Registro automático en MLflow


In [5]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)

if project_root not in sys.path:
    sys.path.append(project_root)

In [30]:
%load_ext autoreload
%autoreload 2

In [15]:
import yaml
import mlflow
import pandas as pd
from src.models.train import train_all_models
from src.data.load_data import load_raw_data

with open(f'{project_root}/configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg

{'data': {'path': '/teamspace/studios/this_studio/data/raw/credit.csv',
  'target_col': 'default'},
 'training': {'test_size': 0.3,
  'random_state': 42,
  'experiment_name': 'credit_risk_experiment'}}

In [24]:
df = load_raw_data(cfg['data']['path'])
df.head()

,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_duration,percent_of_income,years_at_residence,age,other_credit,housing,existing_loans_count,job,dependents,phone,default
0,< 0 DM,6,critical,furniture/appliances,1169,unknown,> 7 years,4,4,67,none,own,2,skilled,1,yes,no
1,1 - 200 DM,48,good,furniture/appliances,5951,< 100 DM,1 - 4 years,2,2,22,none,own,1,skilled,1,no,yes
2,unknown,12,critical,education,2096,< 100 DM,4 - 7 years,2,3,49,none,own,1,unskilled,2,no,no
3,< 0 DM,42,good,furniture/appliances,7882,< 100 DM,4 - 7 years,2,4,45,none,other,1,skilled,2,no,no
4,< 0 DM,24,poor,car,4870,< 100 DM,1 - 4 years,3,4,53,none,other,2,skilled,2,no,yes


In [38]:
from src.features.preprocessing import build_preprocessor
from src.features.feature_selection import HybridFeatureSelector

target = cfg["data"]["target_col"]
preprocessor = build_preprocessor(df, target)

df[target] = df[target].replace({'no': 0, 'yes': 1})

X = df.drop(columns=[target])
y = df[target]

# --- Transformar X a matriz numérica ---
X_processed = preprocessor.fit_transform(X)

# Recuperar nombres de columnas después del Ordinal
cat_cols = preprocessor.transformers_[1][2]   # las columnas categóricas originales
num_cols = preprocessor.named_transformers_["num"].feature_names_in_
feature_names = list(num_cols) + list(cat_cols)


In [39]:
feature_names

['months_loan_duration',
 'amount',
 'percent_of_income',
 'years_at_residence',
 'age',
 'existing_loans_count',
 'dependents',
 'checking_balance',
 'credit_history',
 'purpose',
 'savings_balance',
 'employment_duration',
 'other_credit',
 'housing',
 'job',
 'phone']

In [40]:
# Convertir a DataFrame para el selector
X_df = pd.DataFrame(X_processed, columns=feature_names)

selector = HybridFeatureSelector(top_k=10)
selector.fit(X_df, y)

pd.DataFrame({"feature": selector.selected_features_})

,feature
0,checking_balance
1,months_loan_duration
2,credit_history
3,percent_of_income
4,phone
5,amount
6,savings_balance
7,other_credit
8,age
9,housing


## Entrenar modelos completos con MLflow

In [41]:
mlflow.set_experiment(cfg['training']['experiment_name'])

with mlflow.start_run(run_name='notebook_orchestrator'):
    results = train_all_models(
        data_path=cfg['data']['path'],
        target_col=cfg['data']['target_col'],
        test_size=cfg['training']['test_size'],
        random_state=cfg['training']['random_state']
    )

results

2025/12/08 15:57:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/08 15:57:56 INFO mlflow.store.db.utils: Updating database tables
2025/12/08 15:57:56 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/08 15:57:56 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/08 15:57:56 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2025/12/08 15:57:56 INFO alembic.runtime.migration: Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025/12/08 15:57:56 INFO alembic.runtime.migration: Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025/12/08 15:57:56 INFO alembic.runtime.migration: Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025/12/08 15:57:56 INFO alembic.runtime.migration: Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2025/12/08 15:57:56 INFO alembic.runtime.migration: Running 

{'logistic_regression': {'test_auc': 0.7412698412698412,
  'test_accuracy': 0.75,
  'test_f1': 0.4444444444444444,
  'confusion_matrix': [[195, 15], [60, 30]],
  'cv_auc_mean': 0.7255102040816326},
 'xgboost': {'test_auc': 0.7759259259259259,
  'test_accuracy': 0.74,
  'test_f1': 0.49350649350649345,
  'confusion_matrix': [[184, 26], [52, 38]],
  'cv_auc_mean': 0.7573372206025268},
 'mlp': {'test_auc': 0.5458730158730158,
  'test_accuracy': 0.7,
  'test_f1': 0.02173913043478261,
  'confusion_matrix': [[209, 1], [89, 1]],
  'cv_auc_mean': 0.6042759961127309}}